# 05 — Build Global BM25 Index (Tantivy)

This notebook only orchestrates index construction.

The implementation lives in:

`src/rag/retrieval/bm25.py`

Why Tantivy:

- one **global** BM25 index, so IDF statistics are consistent;
- disk-backed inverted index;
- bounded indexing memory via `writer_heap_size`;
- incremental segment writing;
- fast query-time BM25 without scanning every document.


In [1]:
from pathlib import Path
import sys
import time

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.rag.preprocessing.processor import TextProcessor
from src.rag.retrieval.bm25 import BM25Retriever

try:
    import tantivy
    print("Tantivy:", tantivy.__version__)
except ImportError as exc:
    raise ImportError(
        "Install Tantivy first: pip install tantivy==0.26.0"
    ) from exc


Tantivy: tantivy v0.26.0, index_format v7


In [2]:
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "comments_clean.parquet"
)

INDEX_DIR = (
    PROJECT_ROOT
    / "data"
    / "indexes"
    / "product_comments_bm25_tantivy"
)

BATCH_SIZE = 50_000

# Tantivy indexing memory budget.
# Lower this to 64_000_000 if RAM is tight.
WRITER_HEAP_SIZE = 128_000_000

NUM_THREADS = 1
COMMIT_EVERY_BATCHES = 10
OVERWRITE = True

print("Input:", DATA_PATH)
print("Output:", INDEX_DIR)


Input: /home/ali/Desktop/projects/digikala-ai-assistant/data/processed/comments_clean.parquet
Output: /home/ali/Desktop/projects/digikala-ai-assistant/data/indexes/product_comments_bm25_tantivy


In [3]:
processor = TextProcessor()

start = time.perf_counter()

manifest = BM25Retriever.build_from_parquet(
    input_path=DATA_PATH,
    output_path=INDEX_DIR,
    processor=processor,
    batch_size=BATCH_SIZE,
    writer_heap_size=WRITER_HEAP_SIZE,
    num_threads=NUM_THREADS,
    commit_every_batches=COMMIT_EVERY_BATCHES,
    overwrite=OVERWRITE,
)

elapsed = time.perf_counter() - start

print()
print("Build finished.")
print("Elapsed minutes:", round(elapsed / 60, 2))
print(manifest)


Indexed 50,000 documents
Indexed 100,000 documents
Indexed 150,000 documents
Indexed 200,000 documents
Indexed 250,000 documents
Indexed 300,000 documents
Indexed 350,000 documents
Indexed 400,000 documents
Indexed 450,000 documents
Indexed 500,000 documents
Indexed 550,000 documents
Indexed 600,000 documents
Indexed 650,000 documents
Indexed 700,000 documents
Indexed 750,000 documents
Indexed 800,000 documents
Indexed 850,000 documents
Indexed 900,000 documents
Indexed 950,000 documents
Indexed 1,000,000 documents
Indexed 1,050,000 documents
Indexed 1,100,000 documents
Indexed 1,150,000 documents
Indexed 1,200,000 documents
Indexed 1,250,000 documents
Indexed 1,300,000 documents
Indexed 1,350,000 documents
Indexed 1,400,000 documents
Indexed 1,450,000 documents
Indexed 1,500,000 documents
Indexed 1,550,000 documents
Indexed 1,600,000 documents
Indexed 1,650,000 documents
Indexed 1,700,000 documents
Indexed 1,750,000 documents
Indexed 1,800,000 documents
Indexed 1,850,000 documents
Ind

## Load and smoke-test the production retriever

This uses the exact same `BM25Retriever` class used by the RAG system and evaluation.


In [4]:
bm25 = BM25Retriever(
    processor=processor
)

load_start = time.perf_counter()
bm25.load(INDEX_DIR)
load_ms = (time.perf_counter() - load_start) * 1000

print("Documents:", len(bm25.documents))
print("Load latency (ms):", round(load_ms, 2))


Documents: 6153060
Load latency (ms): 379.96


In [5]:
queries = [
    "ضد آفتاب پوست چرب جوش",
    "شامپو ضد ریزش مو",
    "کرم سبک زود جذب",
]

for query in queries:
    start = time.perf_counter()

    results = bm25.retrieve(
        query,
        top_k=5,
    )

    latency_ms = (
        time.perf_counter()
        - start
    ) * 1000

    print()
    print("Query:", query)
    print("Latency (ms):", round(latency_ms, 2))

    display(
        results[
            [
                "id",
                "score",
                "body",
            ]
        ]
    )



Query: ضد آفتاب پوست چرب جوش
Latency (ms): 118.06


,id,score,body
0,49911806,28.221926,بهترین ضد آفتاب پوست چرب
1,47889871,27.511051,ضد آفتاب برای پوست چرب و رنگی
2,53247500,27.038158,ضد آفتاب مناسبی برای پوست چرب
3,36595710,27.038158,برای پوست چرب عالیه
4,39120109,27.038158,ضد آفتاب مناسب پوست های چرب



Query: شامپو ضد ریزش مو
Latency (ms): 84.88


,id,score,body
0,47383270,28.925093,شامپو ضد ریزش مو سبغ ۱۲۰ میلی
1,47761459,28.470936,شامپوی ضد ریزش خوبیه
2,34305804,28.147827,بهترین شامپو
3,17389862,27.130844,بهترین شامپو ضد ریزش مو
4,20499753,26.492588,شامپو تقویت کننده و ضد ریزش مو کپیدرما



Query: کرم سبک زود جذب
Latency (ms): 85.54


,id,score,body
0,50730820,26.603472,کرم سبک و زود جذب
1,54457131,26.179993,کرم سبک و زود جذب
2,33014565,26.179993,کرم زود جذب و سبک
3,44193015,26.179993,زود جذب و سبک
4,13455391,25.081875,کرم سبک زود جذب و خوبیه
